Работа классификации обнаружения аномалий

In [8]:
# Загрузка библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten

In [4]:
# Загрузка даннных 

df = pd.read_csv('mpi_roof.csv', parse_dates=['Date Time'], index_col='Date Time')
n_rows, n_cols = df.shape
print(f'Размер выборки: {n_rows} строк на {n_cols} столбцов')

Размер выборки: 1051920 строк на 20 столбцов


In [15]:

def check_stationarity(series, signif=0.05, name='Ряд'):
    """
    Выполняет ADF и KPSS тесты для проверки стационарности ряда
    """
    print(f'Результаты проверки для: {name}')

    # --- ADF тест ---
    adf_result = adfuller(series, autolag='AIC')
    adf_pvalue = adf_result[1]
    print(f'  ADF: p-value = {adf_pvalue:.5f}')

    # --- KPSS тест ---
    kpss_result = kpss(series, regression='c', nlags='auto')
    kpss_pvalue = kpss_result[1]
    print(f'  KPSS: p-value = {kpss_pvalue:.5f}')

    # --- Интерпретация ---
    adf_stationary = adf_pvalue < signif
    kpss_stationary = kpss_pvalue > signif

    if adf_stationary and kpss_stationary:
        print(f'  >>> Вывод: Ряд "{name}" СТАЦИОНАРЕН')
    elif (not adf_stationary) and (not kpss_stationary):
        print(f'  >>> Вывод: Ряд "{name}" НЕСТАЦИОНАРЕН (единичный корень)')
    elif adf_stationary and (not kpss_stationary):
        print(f'  >>> Вывод: Ряд "{name}" НЕСТАЦИОНАРЕН (возможно, детерминированный тренд)')
    else:
        print(f'  >>> Вывод: Данных недостаточно для вывода о стационарности ряда "{name}"')

check_stationarity(df['T (degC)'], name='Температура')

Результаты проверки для: Температура


MemoryError: 